In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import sklearn

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [6]:
#The first step is to inspect which variables should be used as features for prediction and which variables should be left out.

credit_fraud = pd.read_csv('../data/credit_card_fraud_dataset.csv')
credit_fraud.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0


In [7]:
credit_fraud.shape

(100000, 7)

In [8]:
credit_fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   TransactionID    100000 non-null  int64  
 1   TransactionDate  100000 non-null  object 
 2   Amount           100000 non-null  float64
 3   MerchantID       100000 non-null  int64  
 4   TransactionType  100000 non-null  object 
 5   Location         100000 non-null  object 
 6   IsFraud          100000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 5.3+ MB


In [9]:
credit_fraud.nunique().sort_values()

IsFraud                 2
TransactionType         2
Location               10
MerchantID           1000
Amount              90621
TransactionDate    100000
TransactionID      100000
dtype: int64

In [10]:
#Now before we choose the features, we need to know how the categorical and target variables look like.

credit_fraud['TransactionType'].value_counts()

TransactionType
refund      50131
purchase    49869
Name: count, dtype: int64

In [11]:
credit_fraud['Location'].value_counts()

Location
Chicago         10193
San Diego       10111
Dallas          10076
San Antonio     10062
New York         9993
Houston          9991
Phoenix          9960
Los Angeles      9936
Philadelphia     9873
San Jose         9805
Name: count, dtype: int64

In [12]:
credit_fraud['IsFraud'].value_counts(normalize=True)*100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [13]:
credit_fraud.groupby('MerchantID')['IsFraud'].agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False).head(10)

,count,sum,mean
MerchantID,,,
640,86,5,0.058140
156,103,5,0.048544
583,106,5,0.047170
659,88,4,0.045455
939,91,4,0.043956
436,96,4,0.041667
568,97,4,0.041237
968,73,3,0.041096
401,98,4,0.040816


Conclusion: For the features, we will leave out merchant ID and TransactionID from the features, including the target variable IsFraud.

In [16]:
#Now let's work on transaction date and time. We will convert the transaction date and time into a datetime object and then extract the hour, day, month, and year from it.

credit_fraud['TransactionDate'] = pd.to_datetime(credit_fraud['TransactionDate'])
credit_fraud['TransactionHour'] = credit_fraud['TransactionDate'].dt.hour
credit_fraud['TransactionDay'] = credit_fraud['TransactionDate'].dt.day_name()
credit_fraud['TransactionMonth'] = credit_fraud['TransactionDate'].dt.month_name()
credit_fraud['TransactionYear'] = credit_fraud['TransactionDate'].dt.year

credit_fraud.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud,TransactionHour,TransactionDay,TransactionMonth,TransactionYear
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0,14,Wednesday,April,2024
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0,13,Tuesday,March,2024
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0,10,Monday,January,2024
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0,23,Saturday,April,2024
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0,18,Friday,July,2024


In [19]:
credit_fraud_new = credit_fraud.copy()

In [20]:
credit_fraud_new 


,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud,TransactionHour,TransactionDay,TransactionMonth,TransactionYear
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0,14,Wednesday,April,2024
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0,13,Tuesday,March,2024
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0,10,Monday,January,2024
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0,23,Saturday,April,2024
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0,18,Friday,July,2024
...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,2024-06-07 00:57:36.027591,1057.29,289,refund,San Antonio,0,0,Friday,June,2024
99996,99997,2023-10-22 23:12:36.027594,297.25,745,refund,San Antonio,0,23,Sunday,October,2023
99997,99998,2024-05-31 19:27:36.027597,3448.56,690,purchase,San Antonio,0,19,Friday,May,2024
99998,99999,2024-10-18 09:43:36.027601,3750.79,644,purchase,Philadelphia,0,9,Friday,October,2024


MODEL TRAINING

In [ ]:
#Split the data into training and testing sets.

x = credit_fraud_new.drop(['IsFraud', 'TransactionDate', 'MerchantID', 'TransactionID'], axis=1)
y = credit_fraud_new['IsFraud']

In [23]:
x.shape, y.shape

((100000, 7), (100000,))

In [25]:
#Train/Test Split the dataset into training and testing sets. We will use 80% of the data for training and 20% for testing.

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

#test size of 0.2 means that 20% of the data will be used for testing and 80% for training. 
#The random_state parameter is set to 42 to ensure that the split is reproducible. 
#The stratify parameter is set to y to ensure that the class distribution in the training and testing sets is similar to that of the original dataset (legitimate=99%, fraudulent=1%).

In [30]:
#Verify the split
x_train.shape, x_test.shape

((80000, 7), (20000, 7))

In [31]:
y_train.shape, y_test.shape

((80000,), (20000,))

In [33]:
#Check the class distribution in the training and testing sets to ensure that they are similar to that of the original dataset.
y_train.value_counts(normalize=True) * 100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [34]:
y_test.value_counts(normalize=True) * 100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [35]:
#Check the sum of fraud counts in the training and testing sets to ensure that they are similar to that of the original dataset.
print("Training fraud cases:", y_train.sum())

Training fraud cases: 800


In [36]:
print("Testing fraud cases:", y_test.sum())

Testing fraud cases: 200


PREPROCESSING

In [37]:
#Here we will decide how each feature will be processed before building the model pipeline. 
#We will use one-hot encoding for categorical variables and standard scaling for numerical variables.

x_train.nunique() 

Amount              73953
TransactionType         2
Location               10
TransactionHour        24
TransactionDay          7
TransactionMonth       12
TransactionYear         2
dtype: int64

In [44]:
numeric_features = ['Amount', 'TransactionHour', 'TransactionYear']
categorical_features = ['TransactionType', 'Location', 'TransactionDay', 'TransactionMonth']

In [45]:
#ColumnTransformer allows us to say: 
#"Apply this transformation to these columns, and another transformation to those columns."
#This is much better than manually encoding columns one by one because it makes the preprocessing reproducible and less prone to leakage.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [46]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

In [47]:
#Fit the processor only on the training data.
x_train_processed = preprocessor.fit_transform(x_train)

In [48]:
#Now transform the test data using the same processor. This is important to ensure that the test data is transformed in the same way as the training data.
x_test_processed = preprocessor.transform(x_test)

In [49]:
#Check the results
x_train_processed.shape, x_test_processed.shape

((80000, 34), (20000, 34))

MODEL CREATION 

We will use Logistic regression because:
- it's relatively simple;
- it's fast to train;
- its behavior is comparatively interpretable;
- it gives us a reference point for more complex models.

In [1]:
#LogisticRegression is a classification algorithm designed to estimate the probability that an observation belongs to a particular class.
from sklearn.linear_model import LogisticRegression

In [2]:
logistic_model = LogisticRegression(random_state=42, max_iter=1000)


Logistic Regression uses an iterative optimization process to find its coefficients.

The default number of iterations may sometimes be insufficient, especially after preprocessing several categorical variables.

So:

*max_iter=1000*

gives the optimizer more room to converge.

It doesn't mean the model necessarily performs 1,000 iterations. It means up to 1,000 iterations are allowed.

*random_state=42*

Again, this helps make the experiment reproducible.

In [3]:
#TRAIN THE MODEL
logistic_model.fit(x_train_processed, y_train)

NameError: name 'x_train_processed' is not defined